# RespiScan Hybrid Model: Dual-Branch Feature Fusion (CWT Scalogram + MFCC)
Notebook ini didedikasikan untuk melatih arsitektur model kustom **Dual-Branch Deep Feature Fusion**. Model ini secara sinergis menggabungkan dua representasi akustik yang berbeda secara simultan: **Citra 2D CWT Scalogram** (melalui cabang CNN MobileNetV2) dan **Vektor 1D MFCC** (melalui cabang MLP Dense). Hasil penggabungan dievaluasi secara klinis pada tingkat pasien menggunakan Majority Voting.

### 1. Inisialisasi Pustaka & Setup GPU
Mengimpor pustaka utama PyTorch, Custom Dataset, dan metrik evaluasi.

In [ ]:
import os
import time
import copy
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix

# Windows crash kernel fix
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# Device Configuration
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"🚀 Model Hybrid berjalan di perangkat: {DEVICE}")

### 2. Custom Dataset Loader untuk Input Ganda (Dual-Input)
Mendefinisikan kelas `DualBranchDataset` yang secara cerdas memuat citra Scalogram 2D dan tensor MFCC `.pt` 1D yang saling berkorespondensi untuk setiap subjek batuk.

In [ ]:
DATA_DIR = 'RespiScan_Train_Ready' 
MFCC_DIR = 'RespiScan_MFCC_Only'
PATH_META = r"Metadata.csv"

class DualBranchDataset(Dataset):
    """Dataset kustom untuk memuat citra Scalogram 2D dan tensor MFCC 1D secara bersamaan."""
    def __init__(self, base_dir, mfcc_dir, split, image_transforms=None):
        self.image_dir = os.path.join(base_dir, split)
        self.mfcc_dir = os.path.join(mfcc_dir, split)
        self.image_transforms = image_transforms
        
        self.samples = []
        # Petakan class folder ke label numerik (Healthy -> 0, TB -> 1)
        self.class_map = {'Healthy': 0, 'TB': 1}
        
        for class_name, label in self.class_map.items():
            class_img_dir = os.path.join(self.image_dir, class_name)
            if not os.path.exists(class_img_dir):
                continue
            for f in os.listdir(class_img_dir):
                if f.endswith('.png') and '_clean.png' in f.lower():
                    img_path = os.path.join(class_img_dir, f)
                    # Temukan padanan MFCC tensor file
                    mfcc_f = f.replace('_clean.png', '_clean_mfcc.pt')
                    mfcc_path = os.path.join(self.mfcc_dir, class_name, mfcc_f)
                    
                    if os.path.exists(mfcc_path):
                        self.samples.append((img_path, mfcc_path, label))
                        
    def __len__(self):
        return len(self.samples)
        
    def __getitem__(self, idx):
        img_path, mfcc_path, label = self.samples[idx]
        from PIL import Image
        
        # 1. Load Scalogram
        img = Image.open(img_path).convert('RGB')
        if self.image_transforms:
            img = self.image_transforms(img)
            
        # 2. Load MFCC Tensor (.pt)
        mfcc = torch.load(mfcc_path, weights_only=True).squeeze()
        if len(mfcc.shape) > 1:
            mfcc_feat = torch.mean(mfcc, dim=1) # Rata-rata sumbu waktu (13-dim)
        else:
            mfcc_feat = mfcc
            
        return img, mfcc_feat, label

print("✅ Custom Dual-Input Dataset Loader siap digunakan.")

### 3. Arsitektur Model Dual-Branch Feature Fusion
Mendefinisikan arsitektur jaringan saraf tiruan kustom `DualBranchFusionNet` yang menggabungkan ekstraktor MobileNetV2 (untuk citra Scalogram) dan MLP Dense (untuk MFCC) di layer Concatenation akhir sebelum klasifikasi.

In [ ]:
class DualBranchFusionNet(nn.Module):
    """Arsitektur Model Hybrid Fusion: MobileNetV2 + MLP."""
    def __init__(self):
        super(DualBranchFusionNet, self).__init__()
        
        # Cabang A: Citra Scalogram 2D (MobileNetV2 Feature Extractor)
        self.scalogram_branch = models.mobilenet_v2(weights='DEFAULT')
        num_ftrs = self.scalogram_branch.classifier[1].in_features
        self.scalogram_branch.classifier = nn.Sequential(
            nn.Linear(num_ftrs, 256),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        
        # Cabang B: Vektor Akustik 1D MFCC (Dense MLP)
        self.mfcc_branch = nn.Sequential(
            nn.Linear(13, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU()
        )
        
        # Cabang Utama Fusion (Penggabung Fitur 256 + 32 = 288)
        self.fusion_classifier = nn.Sequential(
            nn.Linear(288, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 2)
        )

    def forward(self, img, mfcc):
        feat_img = self.scalogram_branch(img)
        feat_mfcc = self.mfcc_branch(mfcc)
        
        # Concatenate fitur
        feat_fusion = torch.cat((feat_img, feat_mfcc), dim=1)
        out = self.fusion_classifier(feat_fusion)
        return out

print("✅ Arsitektur Model Hybrid Dual-Branch Fusion siap digunakan.")

### 4. Pelatihan Model Hybrid dengan Weighted Loss & Early Stopping
Proses training model Fusion selama 20 epoch menggunakan Early Stopping dan LR Scheduler otomatis.

In [ ]:
img_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

# Inisialisasi Dataset Loader
train_dataset = DualBranchDataset(DATA_DIR, MFCC_DIR, 'train', img_transforms['train'])
val_dataset = DualBranchDataset(DATA_DIR, MFCC_DIR, 'val', img_transforms['val'])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0)

model = DualBranchFusionNet().to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
weights = torch.tensor([2.0, 1.0]).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=weights)
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

# Loop Training
num_epochs = 20
best_f1 = 0.0
best_model_wts = copy.deepcopy(model.state_dict())

print("🚀 Memulai pelatihan model Hybrid...")
for epoch in range(num_epochs):
    print(f"Epoch {epoch+1}/{num_epochs}")
    print("-" * 15)
    
    for phase in ['train', 'val']:
        if phase == 'train': 
            model.train()
            loader = train_loader
            dataset_len = len(train_dataset)
        else: 
            model.eval()
            loader = val_loader
            dataset_len = len(val_dataset)
            
        running_loss = 0.0
        running_corrects = 0
        all_preds, all_labels = [], []
        
        for imgs, mfccs, labels in loader:
            imgs, mfccs, labels = imgs.to(DEVICE), mfccs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            
            with torch.set_grad_enabled(phase == 'train'):
                outputs = model(imgs, mfccs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)
                
                if phase == 'train':
                    loss.backward()
                    optimizer.step()
                    
            running_loss += loss.item() * imgs.size(0)
            running_corrects += torch.sum(preds == labels.data)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
        epoch_loss = running_loss / dataset_len
        epoch_acc = running_corrects.double() / dataset_len
        epoch_f1 = f1_score(all_labels, all_preds, zero_division=0)
        
        print(f"{phase.upper()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f} F1: {epoch_f1:.4f}")
        
        if phase == 'val':
            scheduler.step(epoch_f1)
            if epoch_f1 > best_f1:
                best_f1 = epoch_f1
                best_model_wts = copy.deepcopy(model.state_dict())
                torch.save(model.state_dict(), 'best_RespiScan_DualBranch_Fusion.pth')
                print("⭐️ Model Hybrid Terbaik Disimpan!")
    print()
    
print(f"🏆 Best Validation F1-Score: {best_f1:.4f}")
model.load_state_dict(best_model_wts)

### 5. Evaluasi Tingkat Pasien Model Hybrid (Majority Vote)
Menguji model Hybrid terbaik pada data TEST menggunakan penggabungan suara mayoritas klinis pasien.

In [ ]:
print("🏥 Memulai Evaluasi Tingkat Pasien model Hybrid di TEST set...")
test_dataset = DualBranchDataset(DATA_DIR, MFCC_DIR, 'test', img_transforms['val'])
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

df_meta = pd.read_csv(PATH_META)
df_meta.columns = df_meta.columns.str.strip()
df_meta['subject'] = df_meta['subject'].astype(str).str.strip()

# Helper to map PID_32A_now_C -> PID_32A
def get_clean_subject_id(subj):
    import re
    return re.sub(r'_now_C$', '', subj)
df_meta['clean_subject'] = df_meta['subject'].apply(get_clean_subject_id)
valid_subjects = sorted(df_meta['clean_subject'].unique(), key=len, reverse=True)

def ekstrak_subject(filename):
    import re
    match = re.search(r'(PID_\d+[AC])', filename)
    if match:
        return match.group(1)
    for subj in valid_subjects:
        if subj in filename: return subj
    return "TIDAK_KETEMU"

results = []
# Mendapatkan path file asli
file_paths = [path for path, _, label in test_dataset.samples]

model.eval()
with torch.no_grad():
    start_idx = 0
    for imgs, mfccs, labels in test_loader:
        imgs, mfccs = imgs.to(DEVICE), mfccs.to(DEVICE)
        outputs = model(imgs, mfccs)
        _, preds = torch.max(outputs, 1)
        
        for j in range(len(preds)):
            current_idx = start_idx + j
            nama_file = os.path.basename(file_paths[current_idx])
            id_subject = ekstrak_subject(nama_file)
            
            results.append({
                'subject': id_subject, 
                'Label_Asli': 'TB' if labels[j].item() == 1 else 'Healthy',
                'Tebakan_AI': 'TB' if preds[j].item() == 1 else 'Healthy'
            })
        start_idx += len(imgs) # Fixed bug: len(inputs) -> len(imgs)

df_pred = pd.DataFrame(results)
df_valid = df_pred[df_pred['subject'] != 'TIDAK_KETEMU']
grouped = df_valid.groupby('subject')

patient_results = []
for subject, group in grouped:
    true_label = group['Label_Asli'].iloc[0]
    majority_vote = group['Tebakan_AI'].value_counts().index[0]
    patient_results.append({
        'ID_Pasien': subject,
        'Label_Asli': true_label,
        'Keputusan_AI': majority_vote,
        'Benar': true_label == majority_vote
    })

df_patient = pd.DataFrame(patient_results)
y_true = df_patient['Label_Asli']
y_pred = df_patient['Keputusan_AI']

accuracy = accuracy_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred, labels=['Healthy', 'TB'])
tn, fp, fn, tp = cm.ravel()
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

print("\n" + "="*50)
print("      📊 HASIL EVALUASI TINGKAT PASIEN MODEL HYBRID        ")
print("="*50)
print(f"Total Pasien Diuji   : {len(df_patient)} Orang")
print(f"Akurasi Pasien Hybrid: {accuracy * 100:.2f}%")
print(f"Sensitivitas (TB)    : {sensitivity * 100:.2f}%")
print(f"Spesifisitas (Sehat) : {specificity * 100:.2f}%")
print("="*50)